In [1]:
import pandas as pd


In [2]:

data = pd.read_csv("binding_dataset_transformed.csv")

In [3]:
data.head()

,Binding Energy,Otsu Class theshold -77.8,Abs dist to threshold,a_1,a_2,a_3,a_4,a_5,a_6,a_7,a_8,a_9
0,-92.5,1,14.7,1,18,14,9,10,1,20,18,13
1,-77.1,0,0.7,18,14,9,10,1,20,18,13,6
2,-99.6,1,21.8,1,20,18,13,6,13,1,16,10
3,-48.5,0,29.3,14,20,14,14,5,1,5,0,10
4,-50.1,0,27.7,10,10,15,15,11,20,15,18,0


In [6]:
data_true = pd.get_dummies(data,columns=["a_1","a_2","a_3","a_4","a_5","a_6","a_7","a_8","a_9"]).astype(int)

In [7]:
data_true = data_true.drop(columns=["Abs dist to threshold"])

In [8]:
data_true.head()


,Binding Energy,Otsu Class theshold -77.8,a_1_0,a_1_1,a_1_2,a_1_4,a_1_5,a_1_6,a_1_8,a_1_9,...,a_9_7,a_9_8,a_9_9,a_9_10,a_9_12,a_9_13,a_9_14,a_9_15,a_9_18,a_9_20
0,-92,1,0,1,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,-77,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,-99,1,0,1,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,-48,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
4,-50,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
import numpy as np
from cvxopt import matrix, solvers

from sklearn.base import BaseEstimator, ClassifierMixin
class SVM(BaseEstimator, ClassifierMixin):
    """
    Support Vector Machine (SVM) classifier implemented using
    quadratic programming.

    This class supports linear, polynomial, radial basis function (RBF),
    and custom precomputed kernels. It follows the scikit-learn estimator
    interface, allowing compatibility with standard ML workflows.

    Parameters
    ----------
    C : float, default=10
        Regularization parameter that controls the trade-off between
        maximizing the margin and minimizing classification error.

    kernel : str, default='poly'
        Kernel type to be used in the algorithm. Options are:
        'linear', 'poly', 'rbf', or 'Given'.

    r : float, default=1
        Independent term in the polynomial kernel.

    d : int, default=3
        Degree of the polynomial kernel.

    gamma : float, default=0.1
        Kernel coefficient for the RBF kernel.

    k_given : array-like, optional
        Precomputed kernel matrix used when kernel='Given'.

    Attributes
    ----------
    alpha : ndarray
        Lagrange multipliers obtained from the quadratic optimization.

    support_vectors : ndarray
        Subset of training data corresponding to non-zero alpha values.

    w : ndarray or None
        Weight vector (only for linear kernel).

    b : float
        Bias term of the decision function.

    Notes
    -----
    This implementation solves the dual optimization problem using
    quadratic programming via cvxopt.
    """
    def __init__(self, C=10, kernel='poly', r=1, d=3, gamma=0.1,k_given = None):
        self.C = C
        self.kernel = kernel
        self.r = r
        self.d = d
        self.gamma = gamma
        self.alpha = None
        self.support_vectors = None
        self.w = None
        self.b = None
        self.k_given = k_given
    
    def get_params(self, deep=True):
        return {
            'C': self.C,
            'kernel': self.kernel,
            'r': self.r,
            'd': self.d,
            'gamma': self.gamma
        }
    
    def set_params(self, **parameters):
        for parameter, value in parameters.items():
            setattr(self, parameter, value)
        return self
    
    def polynomial_kernel(self, X1, X2):
        return (np.dot(X1, X2.T) + self.r) ** self.d
    
    def rbf_kernel(self, X1, X2):
        sq_dist = np.sum(X1**2, axis=1).reshape(-1, 1) + np.sum(X2**2, axis=1) - 2 * np.dot(X1, X2.T)
        return np.exp(-self.gamma * np.clip(sq_dist, 0, None))
    
    def fit(self, X, y):
        m, n = X.shape
        y = np.where(y <= 0, -1, 1)

        if self.kernel == 'linear':
            K = np.dot(X, X.T)
        elif self.kernel == 'poly':
            K = self.polynomial_kernel(X,X)
        elif self.kernel == 'rbf':
            K = self.rbf_kernel(X, X)
        elif self.kernel == 'Given':
            K= self.k_given
        else:
            raise ValueError("Kernel no soportado")

        P = np.outer(y, y) * K
        P = matrix(P.astype(np.double))
        q = matrix(-np.ones(m).astype(np.double))
        G = matrix(np.vstack((-np.eye(m), np.eye(m))))
        h = matrix(np.hstack((np.zeros(m), np.ones(m) * self.C)))
        A = matrix(y.reshape(1, -1).astype(np.double))
        b = matrix(0.0)

        sol = solvers.qp(P, q, G, h, A, b)
        self.alpha = np.array(sol['x']).flatten()
        sv_indices = self.alpha > 1e-5
        self.support_vectors = X[sv_indices]
        self.alpha_sv = self.alpha[sv_indices]
        self.y_sv = y[sv_indices]
        
        if self.kernel == 'linear':
            self.w = np.sum((self.alpha_sv * self.y_sv).reshape(-1, 1) * self.support_vectors, axis=0)
        else:
            self.w = None
        
        self.b = np.mean(self.y_sv - np.dot(self.support_vectors, self.w)) if self.kernel == 'linear' else np.mean(self.y_sv)
    
    def predict(self, X):
        if self.kernel == 'linear':
            output = np.dot(X, self.w) + self.b
        elif self.kernel == 'poly':
            output = np.sum(
                (self.alpha_sv * self.y_sv).reshape(-1, 1) * 
                (np.dot(self.support_vectors, X.T) + self.r) ** self.d,
                axis=0
            ) + self.b
        elif self.kernel == 'rbf':
            output = np.sum(
                (self.alpha_sv * self.y_sv).reshape(-1, 1) * 
                self.rbf_kernel(self.support_vectors, X),
                axis=0
            ) + self.b
        return np.where(output >= 0, 1, 0)

In [11]:
import numpy as np
import pandas as pd
import cvxopt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split , KFold
from sklearn.metrics import accuracy_score


In [12]:
from sklearn.model_selection import ShuffleSplit
import pandas as pd

X = data.drop(columns=['Otsu Class theshold -77.8'])
y = data['Otsu Class theshold -77.8']

ss = ShuffleSplit(n_splits=4, test_size=0.5, random_state=19)

folds_data = []

for fold, (train_idx, val_idx) in enumerate(ss.split(X, y)):
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]
    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    folds_data.append({
        'fold': fold + 1,
        'X_train': X_train,
        'y_train': y_train,
        'X_val': X_val,
        'y_val': y_val
    })


In [13]:

# Fold 1
fold_1_X_train = folds_data[0]['X_train']
fold_1_y_train = folds_data[0]['y_train']
fold_1_X_val = folds_data[0]['X_val']
fold_1_y_val = folds_data[0]['y_val']
# fold 2
fold_2_X_train = folds_data[1]['X_train']
fold_2_y_train = folds_data[1]['y_train']
fold_2_X_val = folds_data[1]['X_val']
fold_2_y_val = folds_data[1]['y_val']
# fold 3
fold_3_X_train = folds_data[2]['X_train']
fold_3_y_train = folds_data[2]['y_train']
fold_3_X_val = folds_data[2]['X_val']
fold_3_y_val = folds_data[2]['y_val']
# fold 4
fold_4_X_train = folds_data[3]['X_train']
fold_4_y_train = folds_data[3]['y_train']
fold_4_X_val = folds_data[3]['X_val']
fold_4_y_val = folds_data[3]['y_val']

In [14]:
len(fold_1_X_val)

40

In [15]:
scaler = StandardScaler()

#standarize data for training
scaler.fit(fold_1_X_train)
scaler.fit(fold_2_X_train)
scaler.fit(fold_3_X_train)
scaler.fit(fold_4_X_train)

# standarize data for validation
scaler.fit(fold_1_X_val)
scaler.fit(fold_2_X_val)
scaler.fit(fold_3_X_val)
scaler.fit(fold_4_X_val)

StandardScaler()

In [16]:
# standarize variable for training
standar_fold_1_X_train = scaler.transform(fold_1_X_train)
standar_fold_2_X_train = scaler.transform(fold_2_X_train)
standar_fold_3_X_train = scaler.transform(fold_3_X_train)
standar_fold_4_X_train = scaler.transform(fold_4_X_train)
# standarize variable for vaidation
standar_fold_1_X_val = scaler.transform(fold_1_X_val)
standar_fold_2_X_val = scaler.transform(fold_2_X_val)
standar_fold_3_X_val = scaler.transform(fold_3_X_val)
standar_fold_4_X_val = scaler.transform(fold_4_X_val)

In [17]:
print(len(standar_fold_1_X_train))

40


In [18]:
print(len(standar_fold_1_X_val))

40


In [19]:
# classifier = SVM(C = 1 ,kernel = 'rbf', r = 0.2 , d = 2, gamma = 0.1,k_given= None) # best fo 50% rbf

In [20]:
classifier = SVM(C = 1 ,kernel = 'linear', r = 0.2 , d = 2, gamma = 0.01,k_given= None) # best fo 50% linear

In [21]:
# clasifier for fold 1 with C = 1
classifier.fit(standar_fold_1_X_train , fold_1_y_train)

     pcost       dcost       gap    pres   dres
 0: -1.0523e+01 -7.9699e+01  4e+02  2e+00  2e-15
 1: -6.2205e+00 -4.7708e+01  7e+01  3e-01  2e-15
 2: -3.5818e+00 -1.1216e+01  8e+00  2e-02  3e-15
 3: -4.8868e+00 -6.4492e+00  2e+00  3e-03  2e-15
 4: -5.3213e+00 -5.8004e+00  5e-01  6e-04  2e-15
 5: -5.4922e+00 -5.5742e+00  8e-02  7e-05  2e-15
 6: -5.5194e+00 -5.5330e+00  1e-02  1e-05  2e-15
 7: -5.5246e+00 -5.5249e+00  3e-04  1e-07  1e-15
 8: -5.5247e+00 -5.5247e+00  4e-06  1e-09  2e-15
Optimal solution found.


In [22]:
# accuracy score on the training data (Ein)
X_train_prediction = classifier.predict(standar_fold_1_X_train)
training_data_accuracy = accuracy_score( fold_1_y_train, X_train_prediction)
print('Accuracy score of the test data : ', training_data_accuracy)

Accuracy score of the test data :  0.975


In [23]:
# accuracy score on the validating data (Eout)
X_train_prediction = classifier.predict(standar_fold_1_X_val)
training_data_accuracy_1 = accuracy_score( fold_1_y_val, X_train_prediction)
print('Accuracy score of the test data : ', training_data_accuracy_1)

Accuracy score of the test data :  0.9


In [24]:
# clasifier for fold 2 with C = 1
classifier.fit(standar_fold_2_X_train , fold_2_y_train)

     pcost       dcost       gap    pres   dres
 0: -1.0859e+01 -7.2692e+01  3e+02  2e+00  2e-15
 1: -6.8857e+00 -4.1798e+01  5e+01  2e-01  3e-15
 2: -5.1438e+00 -1.2518e+01  9e+00  2e-02  5e-15
 3: -6.1695e+00 -7.6583e+00  2e+00  4e-03  2e-15
 4: -6.5954e+00 -6.8440e+00  3e-01  6e-04  2e-15
 5: -6.6761e+00 -6.7085e+00  3e-02  5e-05  2e-15
 6: -6.6867e+00 -6.6881e+00  1e-03  2e-06  2e-15
 7: -6.6871e+00 -6.6872e+00  2e-05  2e-08  2e-15
 8: -6.6872e+00 -6.6872e+00  2e-07  2e-10  2e-15
Optimal solution found.


In [25]:
# accuracy score on the training data (Ein)
X_train_prediction = classifier.predict(standar_fold_2_X_train)
training_data_accuracy = accuracy_score( fold_2_y_train, X_train_prediction)
print('Accuracy score of the test data : ', training_data_accuracy)

Accuracy score of the test data :  0.95


In [26]:
# accuracy score on the validating data (Eout)
X_train_prediction = classifier.predict(standar_fold_2_X_val)
training_data_accuracy_2 = accuracy_score( fold_2_y_val, X_train_prediction)
print('Accuracy score of the test data : ', training_data_accuracy_2)

Accuracy score of the test data :  0.875


In [27]:
# clasifier for fold 3 with C = 1
classifier.fit(standar_fold_3_X_train , fold_3_y_train)

     pcost       dcost       gap    pres   dres
 0: -9.9859e+00 -7.4994e+01  3e+02  2e+00  3e-15
 1: -5.9078e+00 -4.2833e+01  6e+01  3e-01  2e-15
 2: -3.8489e+00 -9.8913e+00  7e+00  2e-02  6e-15
 3: -4.9386e+00 -6.0900e+00  1e+00  3e-03  3e-15
 4: -5.2754e+00 -5.4452e+00  2e-01  3e-04  2e-15
 5: -5.3392e+00 -5.3467e+00  8e-03  9e-06  3e-15
 6: -5.3426e+00 -5.3428e+00  2e-04  2e-07  3e-15
 7: -5.3427e+00 -5.3427e+00  3e-06  2e-09  3e-15
Optimal solution found.


In [28]:
# accuracy score on the training data (Ein)
X_train_prediction = classifier.predict(standar_fold_3_X_train)
training_data_accuracy = accuracy_score( fold_3_y_train, X_train_prediction)
print('Accuracy score of the test data : ', training_data_accuracy)

Accuracy score of the test data :  0.975


In [29]:
# accuracy score on the validating data (Eout)
X_train_prediction = classifier.predict(standar_fold_3_X_val)
training_data_accuracy_3 = accuracy_score( fold_3_y_val, X_train_prediction)
print('Accuracy score of the test data : ', training_data_accuracy_3)

Accuracy score of the test data :  0.9


In [30]:
# clasifier for fold 3 with C = 1
classifier.fit(standar_fold_4_X_train , fold_4_y_train)

     pcost       dcost       gap    pres   dres
 0: -1.2119e+01 -8.3774e+01  4e+02  2e+00  2e-15
 1: -7.4956e+00 -5.2081e+01  7e+01  3e-01  3e-15
 2: -6.0499e+00 -1.3950e+01  9e+00  2e-02  6e-15
 3: -7.2746e+00 -8.7571e+00  2e+00  3e-03  3e-15
 4: -7.6832e+00 -8.2670e+00  6e-01  7e-04  2e-15
 5: -7.8106e+00 -7.9605e+00  2e-01  2e-05  3e-15
 6: -7.8622e+00 -7.8710e+00  9e-03  1e-06  2e-15
 7: -7.8653e+00 -7.8655e+00  1e-04  2e-08  2e-15
 8: -7.8654e+00 -7.8654e+00  1e-06  2e-10  2e-15
Optimal solution found.


In [31]:
# accuracy score on the training data (Ein)
X_train_prediction = classifier.predict(standar_fold_4_X_train)
training_data_accuracy = accuracy_score( fold_4_y_train, X_train_prediction)
print('Accuracy score of the test data : ', training_data_accuracy)

Accuracy score of the test data :  0.975


In [32]:
# accuracy score on the validating data (Eout)
X_train_prediction = classifier.predict(standar_fold_4_X_val)
training_data_accuracy_4 = accuracy_score( fold_4_y_val, X_train_prediction)
print('Accuracy score of the test data : ', training_data_accuracy_4)

Accuracy score of the test data :  0.975


In [33]:
CV = training_data_accuracy_4 + training_data_accuracy_3 + training_data_accuracy_2 + training_data_accuracy_1
print("Acording to the cross validation the expected Eout without kernel is == " , 1- CV/4 )
print("So the Accuracy score is == " , CV/4)
val = CV/4*100
valpercent = format(val, ".4g")
print("So we get a" , valpercent, "% Accuracy") # for 20% it is very accurate!

Acording to the cross validation the expected Eout without kernel is ==  0.08750000000000002
So the Accuracy score is ==  0.9125
So we get a 91.25 % Accuracy


In [34]:
# classifier = SVM(C = 1 ,kernel = 'rbf', r = 0.2 , d = 2, gamma = 0.03,k_given= None) # this one showed the best results for 20% training

In [35]:
# classifier = SVM(C = 0.1 ,kernel = 'linear', r = 0.2 , d = 2, gamma = 0.04,k_given= None) # this one showed the best results for 10% training


In [36]:
# classifier = SVM(C = 0.001,kernel = 'rbf', r = 0.2 , d = 2, gamma = 0.004,k_given= None) # this one showed the best results for 5% training


In [37]:
# classifier = SVM(C = 1 ,kernel = 'linear', r = 0.2 , d = 2, gamma = 0.01,k_given= None) # linear kernel

In [38]:
# for rbf
### 81.25% for 50% of the data
### 69.53% for 20% of the data
### 68.40% for 10% of the data
### 57.57% for 5% of the data

In [39]:
# linear
### 91.25 % for 50% of the data
### 71.88% for 20% of the data
### 69.44% for 10% of the data
### 62.50% for 5% of the data